In [ ]:
import os

label_folder = r"D:\MyProjects\makanan\tes\auto_sam\labels"

# Loop semua file .txt di folder label
for label_file in os.listdir(label_folder):
    if label_file.endswith(".txt"):
        label_path = os.path.join(label_folder, label_file)

        # Baca isi file
        with open(label_path, "r") as f:
            lines = f.readlines()

        # Ubah ID (angka pertama) jadi 1
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) > 0:
                parts[0] = "0"
                new_lines.append(" ".join(parts))

        # Simpan kembali
        with open(label_path, "w") as f:
            f.write("\n".join(new_lines))

        print(f"Updated ID to 1 in: {label_file}")

print("Semua ID dalam file label telah diubah menjadi 0.")

In [ ]:
import os
import shutil

# Path folder
label_folder = r"D:\MyProjects\makanan\Dataset_workshop\other\data\label"
image_folder = r"D:\MyProjects\makanan\tes\tempe"
output_folder = r"D:\MyProjects\makanan\Dataset_workshop\other\data\tempe_gambar"

# Bikin folder output kalau belum ada
os.makedirs(output_folder, exist_ok=True)

# Ambil nama file label (tanpa ekstensi)
label_names = {os.path.splitext(f)[0] for f in os.listdir(label_folder) if f.endswith('.txt')}

# Loop semua gambar di folder tempe
for image_name in os.listdir(image_folder):
    name, ext = os.path.splitext(image_name)
    if name in label_names:
        src_path = os.path.join(image_folder, image_name)
        dst_path = os.path.join(output_folder, image_name)
        shutil.copy2(src_path, dst_path)
        print(f"Copied: {image_name}")

print("Selesai menyalin gambar yang sesuai label.")

In [1]:
import os
import cv2
import numpy as np
import albumentations as A
from albumentations import ReplayCompose
from glob import glob
from tqdm import tqdm
from natsort import natsorted  # Untuk natural sort

# =============================================================================
# KONFIGURASI DAN PATH
# =============================================================================

BASE_PATH = r"C:\AI\kue\Dataset_workshop\bengkel"
IMAGE_PATH = os.path.join(BASE_PATH, "kue_gambar")
LABEL_PATH = os.path.join(BASE_PATH, "kue_labels")
OUTPUT_IMAGE_PATH = os.path.join(BASE_PATH, "augmented_image")
OUTPUT_LABEL_PATH = os.path.join(BASE_PATH, "augmented_label")
os.makedirs(OUTPUT_IMAGE_PATH, exist_ok=True)
os.makedirs(OUTPUT_LABEL_PATH, exist_ok=True)

SELECT_IMAGE_START = "kue (1)"  
SELECT_IMAGE_END = "kue (27000)"      
NUM_AUGMENTATIONS_PER_IMAGE = 5
COPY_ORIGINAL_ENABLED = True

# Variabel Global untuk strategi multi-scale/tiling dan padding
MULTISCALE_ENABLED = True
REFLECTED_PADDING_ENABLED = True
TARGET_SIZE = 869  # Resolusi output standar

# =============================================================================
# FUNGSI BANTUAN
# =============================================================================

def polygon_to_mask(img_shape, polygon):
    """
    Membuat mask biner berdasarkan poligon.
    Args:
        img_shape: Tuple (tinggi, lebar, channel) gambar.
        polygon: Array koordinat poligon (N, 2).
    Returns:
        Mask biner dengan ukuran gambar.
    """
    mask = np.zeros(img_shape[:2], dtype=np.uint8)
    pts = polygon.reshape((-1, 1, 2)).astype(np.int32)
    cv2.fillPoly(mask, [pts], 255)
    return mask

def ensure_clockwise(pts):
    """
    Pastikan titik poligon berurutan searah jarum jam.
    Jika area (metode Shoelace) positif (artinya berurutan counter-clockwise), balik urutannya.
    """
    area = 0
    for i in range(len(pts)):
        j = (i + 1) % len(pts)
        area += pts[i][0] * pts[j][1] - pts[j][0] * pts[i][1]
    if area > 0:
        pts = pts[::-1]
    return pts

def remove_duplicate_labels(label_data, precision=4):
    """
    Menghapus duplikasi label berdasarkan class_id dan koordinat poligon.
    Label dianggap duplikat jika class_id-nya sama dan koordinat poligon (setelah pembulatan)
    identik.
    """
    unique = {}
    for class_id, pts in label_data:
        # Bulatkan koordinat untuk menghindari perbedaan kecil
        rounded = tuple(np.round(pts.flatten(), precision))
        key = (class_id, rounded)
        unique[key] = (class_id, pts)
    return list(unique.values())

# =============================================================================
# PIPELINE AUGMENTASI (ReplayCompose)
# =============================================================================

# Tentukan border_mode berdasarkan opsi padding
border_mode = cv2.BORDER_REFLECT if REFLECTED_PADDING_ENABLED else cv2.BORDER_CONSTANT

# Jika multi-scale diaktifkan, tambahkan transformasi skala acak
scale_transforms = []
if MULTISCALE_ENABLED:
    scale_transforms.append(A.RandomScale(scale_limit=(0.5, 1.5), p=1.0))

transform_list = scale_transforms + [
    A.HorizontalFlip(p=0.5),               # flip 50%
    A.VerticalFlip(p=0.5),                 # vertical jarang tapi perlu
    A.RandomBrightnessContrast(0.1, 0.1, p=0.3),
    A.HueSaturationValue(5, 10, 10, p=0.3),
    A.Rotate(limit=15, p=0.5),             # rotasi ringan
    A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.05), shear=5, p=0.5),
    A.GaussNoise(std_range=(0.1, 0.2), p=0.2),
    A.MotionBlur(blur_limit=5, p=0.2),
    A.CLAHE(p=0.2),                         # bantu objek low-contrast
    A.LongestMaxSize(max_size=TARGET_SIZE),
    A.PadIfNeeded(min_height=TARGET_SIZE, min_width=TARGET_SIZE, border_mode=border_mode)
]

replay_pipeline = ReplayCompose(transform_list)

# =============================================================================
# MEMBACA FILE GAMBAR DAN LABEL
# =============================================================================

# Buat dictionary dengan key berupa nama file tanpa ekstensi
image_files = {os.path.splitext(os.path.basename(f))[0]: f 
               for f in glob(os.path.join(IMAGE_PATH, "*.jpg"))}
label_files = {os.path.splitext(os.path.basename(f))[0]: f 
               for f in glob(os.path.join(LABEL_PATH, "*.txt"))}

# Buat list sorted_keys berdasarkan file asli
sorted_keys = natsorted(list(image_files.keys()))

# Konversi list dan range key ke lowercase untuk perbandingan
sorted_keys_lower = [key.lower() for key in sorted_keys]
start_key = SELECT_IMAGE_START.lower()
end_key = SELECT_IMAGE_END.lower()

try:
    start_idx = sorted_keys_lower.index(start_key)
    end_idx = sorted_keys_lower.index(end_key) + 1
    selected_keys = sorted_keys[start_idx:end_idx]
except ValueError as e:
    print(f"Range tidak ditemukan: pastikan '{SELECT_IMAGE_START}' dan '{SELECT_IMAGE_END}' ada di dalam daftar file.")
    selected_keys = sorted_keys  # Jika range tidak ditemukan, gunakan semua file

# =============================================================================
# PROSES AUGMENTASI SECARA BATCH PER GAMBAR
# =============================================================================

def process_image_batch(filename):
    """
    Proses augmentasi untuk satu gambar beserta label-nya.
    Melakukan augmentasi berulang kali dan menyesuaikan koordinat poligon.
    """
    img_file = image_files.get(filename)
    lbl_file = label_files.get(filename)
    if not img_file or not lbl_file:
        return

    image = cv2.imread(img_file)
    if image is None:
        return
    H, W = image.shape[:2]

    # Baca dan parsing label (format: "class_id x0 y0 x1 y1 ...")
    with open(lbl_file, "r") as f:
        lines = f.readlines()

    polygons = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        class_id = parts[0]
        coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
        # Konversi koordinat relatif ke piksel
        coords[:, 0] *= W
        coords[:, 1] *= H
        polygons.append((class_id, coords))

    if not polygons:
        return

    # Lakukan augmentasi NUM_AUGMENTATIONS_PER_IMAGE kali untuk tiap gambar
    for i in range(NUM_AUGMENTATIONS_PER_IMAGE):
        replay_result = replay_pipeline(image=image)
        aug_image = replay_result["image"]
        replay_params = replay_result["replay"]

        new_label_data = []
        for class_id, poly in polygons:
            mask = polygon_to_mask(image.shape, poly)
            # Ubah mask ke format 3 channel
            mask_3c = np.stack([mask] * 3, axis=-1)
            aug_mask_result = A.ReplayCompose.replay(replay_params, image=mask_3c)["image"]
            aug_mask = cv2.cvtColor(aug_mask_result, cv2.COLOR_BGR2GRAY)
            _, aug_mask = cv2.threshold(aug_mask, 127, 255, cv2.THRESH_BINARY)

            contours, _ = cv2.findContours(aug_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in contours:
                if cv2.contourArea(cnt) < 200:
                    continue
                epsilon = 0.005 * cv2.arcLength(cnt, True)
                approx = cv2.approxPolyDP(cnt, epsilon, True)
                pts = approx.reshape(-1, 2).astype(float)
            
                pts = ensure_clockwise(pts)
                h2, w2 = aug_image.shape[:2]
                pts[:, 0] /= w2
                pts[:, 1] /= h2
                new_label_data.append((class_id, pts))
        # Hapus label duplikat
        new_label_data = remove_duplicate_labels(new_label_data)

        out_img_path = os.path.join(OUTPUT_IMAGE_PATH, f"{filename}_aug_{i+1}.jpg")
        cv2.imwrite(out_img_path, aug_image)
        out_lbl_path = os.path.join(OUTPUT_LABEL_PATH, f"{filename}_aug_{i+1}.txt")
        with open(out_lbl_path, "w") as f:
            for class_id, pts in new_label_data:
                pts_flat = pts.flatten().tolist()
                f.write(f"{class_id} " + " ".join(map(str, pts_flat)) + "\n")

# =============================================================================
# MENYALIN DATA ASLI (ORIGINAL)
# =============================================================================

def copy_original(filename):
    """
    Menyalin file gambar dan label asli ke direktori output dengan suffix '_ori'.
    """
    img_file = image_files.get(filename)
    lbl_file = label_files.get(filename)
    if img_file:
        image = cv2.imread(img_file)
        if image is not None:
            out_img_path = os.path.join(OUTPUT_IMAGE_PATH, f"{filename}_ori.jpg")
            cv2.imwrite(out_img_path, image)
    if lbl_file:
        out_lbl_path = os.path.join(OUTPUT_LABEL_PATH, f"{filename}_ori.txt")
        with open(lbl_file, "r") as f_in, open(out_lbl_path, "w") as f_out:
            f_out.write(f_in.read())

# =============================================================================
# MAIN FUNCTION
# =============================================================================

def main():
    # Salin data asli jika toggle aktif
    if COPY_ORIGINAL_ENABLED:
        for filename in tqdm(selected_keys, desc="Copying Original Images"):
            copy_original(filename)

    # Proses augmentasi untuk tiap gambar yang telah diseleksi
    for filename in tqdm(selected_keys, desc="Augmenting Images"):
        process_image_batch(filename)
    print("Proses augmentasi selesai!")

if __name__ == "__main__":
    main()

C:\Users\Administrator\AppData\Roaming\Python\Python312\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.6' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Range tidak ditemukan: pastikan 'kue (1)' dan 'kue (27000)' ada di dalam daftar file.


Augmenting Images: 100%|██████████| 59/59 [05:27<00:00,  5.55s/it]

Proses augmentasi selesai!


In [ ]:
# === Ekstraksi Objek Polygon YOLO ke PNG Transparan + Copy Label ===
# Versi NO AUGMENT - KEEP POLYGON ASLI - SAVE IMAGE & LABEL TERPISAH

import os
import cv2
import numpy as np
from tqdm import tqdm

# === Path ===
image_dir = r"D:\MyProjects\makanan\Dataset_workshop\other\data\images"
label_dir = r"D:\MyProjects\makanan\Dataset_workshop\other\data\labels"

objects_dir = r"D:\MyProjects\makanan\Dataset_workshop\other\data\Data_sistetis\objects"
labels_output_dir = r"D:\MyProjects\makanan\Dataset_workshop\other\data\Data_sistetis\labels"

os.makedirs(objects_dir, exist_ok=True)
os.makedirs(labels_output_dir, exist_ok=True)

# === Class ID target ===
target_class_ids = [0, 1]  # 0 = bawang, 1 = tempe

# === Ambil Semua Gambar ===
image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

counter = 0

for image_path in tqdm(image_paths, desc="Extracting"):
    filename = os.path.splitext(os.path.basename(image_path))[0]
    label_path = os.path.join(label_dir, f"{filename}.txt")

    if not os.path.exists(label_path):
        print(f"Label not found: {label_path}")
        continue

    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    with open(label_path, "r") as f:
        lines = f.readlines()

    for idx, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) < 7:
            continue  # skip yang tidak cukup point

        class_id = int(parts[0])

        # hanya ambil target
        if class_id not in target_class_ids:
            continue

        # ambil polygon (dalam normalized)
        points = np.array(list(map(float, parts[1:])), dtype=np.float32).reshape(-1, 2)
        points[:, 0] *= w  # denormalisasi x
        points[:, 1] *= h  # denormalisasi y
        points = points.astype(np.int32)

        # buat mask polygon
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [points], 255)

        # masking objek
        obj = cv2.bitwise_and(img, img, mask=mask)

        # crop tight bounding box
        x, y, bw, bh = cv2.boundingRect(points)
        cropped_obj = obj[y:y+bh, x:x+bw]
        cropped_mask = mask[y:y+bh, x:x+bw]

        # buat transparan alpha
        alpha = np.zeros_like(cropped_mask)
        alpha[cropped_mask > 0] = 255
        rgba = cv2.merge([cropped_obj, alpha])

        # === Simpan PNG ===
        save_name = f"{filename}_obj{idx}.png"
        save_path = os.path.join(objects_dir, save_name)
        cv2.imwrite(save_path, rgba)

        # === Simpan Label ke Folder labels ===
        label_save_name = f"{filename}_obj{idx}.txt"
        label_save_path = os.path.join(labels_output_dir, label_save_name)
        with open(label_save_path, "w") as f_out:
            f_out.write(line)  # tetap tulis poligon asli (TANPA PERUBAHAN)

        counter += 1

print(f"\nTotal objek berhasil diekstrak: {counter} (class_id: {target_class_ids})")
print("Selesai.")

In [ ]:
import os
import glob
import random
import cv2
import numpy as np
from tqdm import tqdm

# ========== KONFIGURASI ==========

N_IMAGES = 511
CANVAS_SIZE = (1024, 1024)
OBJ_SCALE_RANGE = (0.1, 0.2)
OBS_SCALE_RANGE = (0.2, 0.3)
N_OBJ_RANGE = (2, 4)
N_OBS_RANGE = (0, 2)
OUTPUT_DIR = r"D:\MyProjects\makanan\Dataset_workshop\other\data\Data_sistetis\synthetic"

IMAGE_DIR = os.path.join(OUTPUT_DIR, "images")
LABEL_DIR = os.path.join(OUTPUT_DIR, "labels")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)

# ========== LOAD GAMBAR ==========

def load_images(folder):
    images = []
    for ext in ('*.jpg', '*.png'):
        for img_path in glob.glob(os.path.join(folder, ext)):
            img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
            if img is not None:
                images.append(img)
    return images

backgrounds = load_images(r"D:\MyProjects\makanan\Dataset_workshop\other\data\Data_sistetis\backgrounds")
obstacles = load_images(r"D:\MyProjects\makanan\Dataset_workshop\other\data\Data_sistetis\obstacles")
bawang_objects = load_images(r"D:\MyProjects\makanan\Dataset_workshop\other\data\Data_sistetis\objects\bawang")
tempe_objects = load_images(r"D:\MyProjects\makanan\Dataset_workshop\other\data\Data_sistetis\objects\tempe")

print(f"Loaded {len(backgrounds)} backgrounds, {len(bawang_objects)} bawang objects, {len(tempe_objects)} tempe objects, {len(obstacles)} obstacles")

if len(backgrounds) == 0 or len(bawang_objects) == 0 or len(tempe_objects) == 0:
    raise Exception("Pastikan folder background dan object bawang, tempe, obstacle tidak kosong dan format gambarnya benar (jpg/png)")

# ========== FUNGSI ==========

def resize_within_canvas(img, scale_range, canvas_size):
    scale = random.uniform(*scale_range)
    new_w, new_h = int(canvas_size[0] * scale), int(canvas_size[1] * scale)
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

def generate_polygon(x, y, w, h):
    return [(x, y), (x + w, y), (x + w, y + h), (x, y + h)]

def polygon_to_yolo(poly, img_w, img_h):
    return [f"{px / img_w:.6f} {py / img_h:.6f}" for px, py in poly]

def paste_obj(bg, obj, position):
    x, y = position
    h, w = obj.shape[:2]
    if obj.shape[2] == 4:
        alpha = obj[:, :, 3] / 255.0
        for c in range(3):
            bg[y:y+h, x:x+w, c] = bg[y:y+h, x:x+w, c] * (1 - alpha) + obj[:, :, c] * alpha
    else:
        bg[y:y+h, x:x+w] = obj

# ========== SYNTHESIS ==========
for idx in tqdm(range(N_IMAGES)):
    bg = random.choice(backgrounds)
    bg = cv2.resize(bg, CANVAS_SIZE)
    img_h, img_w = CANVAS_SIZE
    label_lines = []
    
    num_bawang = random.randint(1, 2)
    num_tempe = random.randint(1, 2)
    
    for class_id, obj_list in enumerate([bawang_objects, tempe_objects]):
        for _ in range(random.randint(1, 2)):
            obj = resize_within_canvas(random.choice(obj_list), OBJ_SCALE_RANGE, CANVAS_SIZE)
            h, w = obj.shape[:2]
            x, y = random.randint(0, img_w - w), random.randint(0, img_h - h)
            paste_obj(bg, obj, (x, y))
            poly = generate_polygon(x, y, w, h)
            label_lines.append(f"{class_id} " + " ".join(polygon_to_yolo(poly, img_w, img_h)))
    
    filename = f"synthetic_{idx:04d}"
    cv2.imwrite(os.path.join(IMAGE_DIR, filename + ".jpg"), bg)
    
    with open(os.path.join(LABEL_DIR, filename + ".txt"), 'w') as f:
        f.write("\n".join(label_lines))

print("✅ Dataset YOLO Segmentation berhasil dibuat dengan anotasi poligon!")

In [2]:
#-----------------------------------------------------------#
############# STEP 02 : Check hasil Augmentasi ##############
#-----------------------------------------------------------#

import cv2              # Untuk manipulasi gambar
import numpy as np      # Untuk operasi array dan numerik
import random           # Untuk pengacakan (misal: memilih gambar acak, warna)
import os               # Untuk operasi file dan folder
import glob             # Untuk pencarian file dengan pola tertentu
import logging
from tqdm import tqdm   # Untuk progress bar di terminal
import shutil

# ============================================
# Konfigurasi Logging
# ============================================
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers.clear()  # Hapus handler default

# Handler untuk menyimpan log ke file (INFO ke atas)
file_handler = logging.FileHandler("proses.log")
file_handler.setLevel(logging.INFO)
file_formatter = logging.Formatter('%(asctime)s %(levelname)s: %(message)s')
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

# Handler untuk menampilkan log ke terminal (WARNING ke atas)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.WARNING)
console_formatter = logging.Formatter('%(levelname)s: %(message)s')
console_handler.setFormatter(console_formatter)
logger.addHandler(console_handler)

# ============================================
# Definisi Folder Sumber dan Tujuan
# ============================================
aug_img_dir = r"C:\AI\kue\Dataset_workshop\bengkel\augmented_image"
aug_label_dir = r"C:\AI\kue\Dataset_workshop\bengkel\augmented_label"
save_dir = r"C:\AI\kue\Dataset_workshop\bengkel\Cek"

# ============================================
# Pastikan Folder Tujuan Ada dan Kosong
# ============================================
if not os.path.exists(save_dir):
    os.makedirs(save_dir, exist_ok=True)
    logger.info("Folder '%s' dibuat.", save_dir)
else:
    # Kosongkan folder cek jika sudah ada
    for filename in os.listdir(save_dir):
        file_path = os.path.join(save_dir, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            logger.warning("Gagal menghapus %s. Error: %s", file_path, e)
    logger.info("Folder '%s' dikosongkan.", save_dir)

# ============================================
# Pengambilan File Gambar
# ============================================
img_files = glob.glob(os.path.join(aug_img_dir, "*.jpg"))

if len(img_files) < 10000000:
    logger.warning("Gambar kurang dari 100! Menampilkan semua yang ada.")
    random_imgs = img_files
else:
    random_imgs = random.sample(img_files, 100)

logger.info("Menampilkan %d gambar untuk pengecekan.", len(random_imgs))

# ============================================
# Fungsi Bantuan
# ============================================

def compute_polygon_area(pts):
    """Menghitung luas poligon menggunakan metode contourArea OpenCV."""
    return cv2.contourArea(pts)

def is_polygon_within_bounds(pts, width, height):
    """Memeriksa apakah semua titik poligon berada dalam batas gambar."""
    for pt in pts.reshape(-1, 2):
        x, y = pt
        if x < 0 or x > width or y < 0 or y > height:
            return False
    return True

def round_polygon(pts, precision=4):
    """Mengembalikan tuple dari koordinat poligon yang sudah dibulatkan."""
    return tuple(np.round(pts.flatten(), precision))

# ============================================
# Proses Pengolahan dan Pengecekan Tiap Gambar
# ============================================
for img_file in tqdm(random_imgs, desc="Memproses gambar"):
    base_name = os.path.splitext(os.path.basename(img_file))[0]
    label_file = os.path.join(aug_label_dir, base_name + ".txt")
    
    if not os.path.exists(label_file):
        logger.warning("Label untuk %s tidak ditemukan, lewati.", base_name)
        continue
    
    image = cv2.imread(img_file)
    if image is None:
        logger.warning("Gagal membaca %s, lewati.", img_file)
        continue
    
    h, w, _ = image.shape
    original_image = image.copy()  # Untuk menggambar anotasi
    
    with open(label_file, "r") as f:
        lines = f.readlines()
    
    # Dictionary untuk mendeteksi duplikasi: key = (class_id, rounded koordinat)
    seen_polygons = {}
    duplicate_found = False
    
    for idx, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) < 3:
            logger.warning("Format label salah pada %s baris %d.", base_name, idx+1)
            continue

        # Parsing label dan koordinat
        class_id = parts[0]
        try:
            coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
        except Exception as e:
            logger.warning("Gagal parsing koordinat pada %s baris %d. Error: %s", base_name, idx+1, e)
            continue
        
        # Konversi koordinat relatif ke piksel
        coords[:, 0] *= w
        coords[:, 1] *= h
        coords = coords.astype(np.int32)
        pts = coords.reshape((-1, 1, 2))
        
        # Periksa apakah poligon berada dalam batas gambar
        if not is_polygon_within_bounds(pts, w, h):
            logger.warning("Poligon pada %s baris %d berada di luar batas gambar.", base_name, idx+1)
            # Tandai dengan warna oranye
            color = (0, 165, 255)
            cv2.polylines(image, [pts], isClosed=True, color=color, thickness=3)
        else:
            # Tandai dengan warna hijau jika valid
            color = (0, 255, 0)
            cv2.polylines(image, [pts], isClosed=True, color=color, thickness=2)
        
        # Periksa area poligon
        area = compute_polygon_area(pts)
        if area < 10:
            logger.warning("Area poligon terlalu kecil (%.2f) pada %s baris %d.", area, base_name, idx+1)
            # Tandai dengan warna biru
            cv2.polylines(image, [pts], isClosed=True, color=(255, 0, 0), thickness=3)
        
        # Cek duplikasi label
        key = (class_id, round_polygon(pts, precision=2))
        if key in seen_polygons:
            logger.warning("Duplikasi label ditemukan pada %s baris %d. Duplikat dengan baris %d.", 
                           base_name, idx+1, seen_polygons[key])
            duplicate_found = True
            # Tandai duplikasi dengan warna merah
            cv2.polylines(image, [pts], isClosed=True, color=(0, 0, 255), thickness=3)
        else:
            seen_polygons[key] = idx+1  # Simpan nomor baris label
        
    # Simpan gambar hasil pengecekan
    save_path = os.path.join(save_dir, base_name + "_checked.jpg")
    cv2.imwrite(save_path, image)
    
    logger.info("Gambar %s telah dicek dan disimpan di %s", base_name + "_checked.jpg", save_dir)
    
    # Jika ditemukan duplikasi, juga simpan gambar asli untuk referensi
    if duplicate_found:
        dup_save_path = os.path.join(save_dir, base_name + "_duplicate.jpg")
        cv2.imwrite(dup_save_path, original_image)
        logger.info("Gambar asli %s juga disimpan sebagai referensi duplikasi.", base_name)

logger.warning("Proses pengecekan selesai!")

Memproses gambar: 100%|██████████| 354/354 [00:10<00:00, 33.24it/s]


In [ ]:
import os

def check_dataset_integrity(image_dir, label_dir):
    image_extensions = {'.jpg', '.jpeg', '.png'}
    label_extension = '.txt'
    
    image_files = {os.path.splitext(f)[0] for f in os.listdir(image_dir) if os.path.splitext(f)[1].lower() in image_extensions}
    label_files = {os.path.splitext(f)[0] for f in os.listdir(label_dir) if f.endswith(label_extension)}
    
    images_without_labels = image_files - label_files
    labels_without_images = label_files - image_files
    
    empty_labels = []
    empty_images = []
    incorrect_labels = []
    duplicate_labels = []

    for lbl in os.listdir(label_dir):
        lbl_path = os.path.join(label_dir, lbl)
        if lbl.endswith(label_extension):
            with open(lbl_path, 'r', encoding='utf-8') as file:
                content = file.readlines()
                content = [line.strip() for line in content if line.strip()]
                
                # Cek label kosong
                if not content:
                    empty_labels.append(lbl)
                
                # Cek format label yang salah
                elif not all(line.split()[0].isdigit() for line in content):
                    incorrect_labels.append(lbl)
                
                # Cek label duplikat
                elif len(set(content)) < len(content):
                    duplicate_labels.append(lbl)

    return images_without_labels, labels_without_images, empty_labels, empty_images, incorrect_labels, duplicate_labels

def delete_invalid_files(image_dir, label_dir, images_without_labels, labels_without_images, empty_labels, empty_images, incorrect_labels, duplicate_labels):
    for img in images_without_labels:
        for ext in ['.jpg', '.jpeg', '.png']:
            img_path = os.path.join(image_dir, img + ext)
            if os.path.exists(img_path):
                os.remove(img_path)
                print(f"Dihapus: {img_path}")

    for lbl in labels_without_images:
        lbl_path = os.path.join(label_dir, lbl + ".txt")
        if os.path.exists(lbl_path):
            os.remove(lbl_path)
            print(f"Dihapus: {lbl_path}")

    for lbl in empty_labels:
        lbl_path = os.path.join(label_dir, lbl)
        os.remove(lbl_path)
        print(f"Dihapus label kosong: {lbl_path}")

    for img in empty_images:
        img_path = os.path.join(image_dir, img)
        os.remove(img_path)
        print(f"Dihapus gambar kosong: {img_path}")

    for lbl in incorrect_labels:
        lbl_path = os.path.join(label_dir, lbl)
        os.remove(lbl_path)
        print(f"Dihapus label tidak valid: {lbl_path}")

    for lbl in duplicate_labels:
        lbl_path = os.path.join(label_dir, lbl)
        os.remove(lbl_path)
        print(f"Dihapus label duplikat: {lbl_path}")

# Direktori dataset
image_dir = r"D:\MyProjects\makanan\Dataset_workshop\data_yolo\Dataset_v6\augmented_images"
label_dir = r"D:\MyProjects\makanan\Dataset_workshop\data_yolo\Dataset_v6\augmented_labels"

# Cek integritas dataset
images_without_labels, labels_without_images, empty_labels, empty_images, incorrect_labels, duplicate_labels = check_dataset_integrity(image_dir, label_dir)

print("Gambar tanpa label:", images_without_labels or "Tidak ada")
print("Label tanpa gambar:", labels_without_images or "Tidak ada")
print("Label kosong:", empty_labels or "Tidak ada")
print("Gambar kosong:", empty_images or "Tidak ada")
print("Label tidak valid:", incorrect_labels or "Tidak ada")
print("Label duplikat:", duplicate_labels or "Tidak ada")

delete_invalid_files(image_dir, label_dir, images_without_labels, labels_without_images, empty_labels, empty_images, incorrect_labels, duplicate_labels)